# Wafer Defect Classification CNN (PyTorch)

WM811K wafer image dataset을 이용해 9개 defect class를 분류하는 CNN 모델입니다. GPU가 사용 가능하면 CUDA로 학습하고, 그렇지 않으면 CPU로 실행됩니다.


## 1. 라이브러리 불러오기

In [ ]:
import copy
import glob
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Dataset, Subset


## 2. 경로와 클래스 설정

In [ ]:
BASE_DIR = Path(r"C:\Users\user\Desktop\WM811k_origin\waferimages")
OUTPUT_DIR = Path(r"C:\Users\user\Documents\Codex\2026-07-05\co\outputs\wafer_pytorch\results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = [
    "Center", "Donut", "Edge-Loc", "Edge-Ring", "Loc",
    "Near-full", "Random", "Scratch", "none"
]

batch_size = 32
n_epoch = 100
learning_rate = 0.001
patience = 15
num_workers = 0


## 3. GPU 확인

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Device:", device)


## 4. 데이터셋 정의

In [ ]:
class WaferDataset(Dataset):
    def __init__(self, folder, class_names, image_size=32):
        self.folder = Path(folder)
        self.class_names = class_names
        self.image_size = image_size
        self.samples = []

        for label, class_name in enumerate(class_names):
            class_dir = self.folder / class_name
            paths = sorted(glob.glob(str(class_dir / "*")))
            print(f"{self.folder.name:8s} | {class_name:10s} | {len(paths):5d}")
            self.samples.extend((path, label) for path in paths)

        if not self.samples:
            raise RuntimeError(f"No images found in: {self.folder}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label = self.samples[index]
        with Image.open(path) as image:
            image = image.convert("RGB").resize((self.image_size, self.image_size))
            array = np.asarray(image, dtype=np.float32) / 255.0

        tensor = torch.from_numpy(array).permute(2, 0, 1)
        return tensor, label

    @property
    def labels(self):
        return np.array([label for _, label in self.samples], dtype=np.int64)


## 5. CNN 모델 정의

In [ ]:
class WaferCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(0.30),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.40),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


## 6. 평가 함수

In [ ]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_labels.extend(labels.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())

    return total_loss / total, correct / total, np.array(all_labels), np.array(all_preds)


## 7. 데이터 로딩 및 Train/Validation 분리

In [ ]:
train_dataset = WaferDataset(BASE_DIR / "training", CLASS_NAMES)
test_dataset = WaferDataset(BASE_DIR / "testing", CLASS_NAMES)

labels = train_dataset.labels
train_indices, val_indices = train_test_split(
    np.arange(len(train_dataset)),
    test_size=0.2,
    random_state=42,
    stratify=labels,
)

train_subset = Subset(train_dataset, train_indices)
val_subset = Subset(train_dataset, val_indices)

print("train:", np.bincount(train_dataset.labels, minlength=len(CLASS_NAMES)))
print("test :", np.bincount(test_dataset.labels, minlength=len(CLASS_NAMES)))


## 8. Class Weight 계산 및 DataLoader 생성

In [ ]:
y_train_split = labels[train_indices]
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(CLASS_NAMES)),
    y=y_train_split,
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=device)
print("class_weight:", dict(enumerate(class_weights.tolist())))

train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=torch.cuda.is_available())


## 9. 모델 학습

In [ ]:
model = WaferCNN(num_classes=len(CLASS_NAMES)).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5, min_lr=1e-5)

best_val_loss = float("inf")
best_state = copy.deepcopy(model.state_dict())
wait = 0
history = []

for epoch in range(1, n_epoch + 1):
    model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for images, labels_batch in train_loader:
        images = images.to(device)
        labels_batch = labels_batch.to(device)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion(outputs, labels_batch)
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * images.size(0)
        train_correct += (outputs.argmax(dim=1) == labels_batch).sum().item()
        train_total += labels_batch.size(0)

    train_loss = train_loss_sum / train_total
    train_acc = train_correct / train_total
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)
    scheduler.step(val_loss)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "learning_rate": optimizer.param_groups[0]["lr"],
    })

    print(
        f"Epoch {epoch:03d}/{n_epoch} "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} "
        f"lr={optimizer.param_groups[0]['lr']:.6f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch}.")
            break

model.load_state_dict(best_state)


## 10. 테스트 평가 및 결과 저장

In [ ]:
test_loss, test_acc, y_true, y_pred = evaluate(model, test_loader, criterion, device)

print("Test loss:", test_loss)
print("Test accuracy:", test_acc)

report = classification_report(y_true, y_pred, target_names=CLASS_NAMES)
print(report)

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, cmap="Blues", xticks_rotation=45)
fig.tight_layout()
confusion_matrix_path = OUTPUT_DIR / "confusion_matrix.png"
fig.savefig(confusion_matrix_path, dpi=160)
plt.show()

model_path = OUTPUT_DIR / "wafer_cnn_pytorch.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "class_names": CLASS_NAMES,
    "image_size": 32,
    "test_accuracy": test_acc,
    "test_loss": test_loss,
}, model_path)

history_path = OUTPUT_DIR / "history.json"
history_path.write_text(json.dumps(history, indent=2), encoding="utf-8")

summary_path = OUTPUT_DIR / "summary.txt"
summary_path.write_text(
    "\n".join([
        f"PyTorch: {torch.__version__}",
        f"CUDA available: {torch.cuda.is_available()}",
        f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}",
        f"Device: {device}",
        f"Test loss: {test_loss}",
        f"Test accuracy: {test_acc}",
        f"Best val acc: {max(item['val_accuracy'] for item in history)}",
        f"Final train acc: {history[-1]['train_accuracy']}",
        f"Final val acc: {history[-1]['val_accuracy']}",
        f"train: {np.bincount(train_dataset.labels, minlength=len(CLASS_NAMES)).tolist()}",
        f"test: {np.bincount(test_dataset.labels, minlength=len(CLASS_NAMES)).tolist()}",
        "",
        report,
    ]),
    encoding="utf-8",
)

print("Saved model:", model_path)
print("Saved confusion matrix:", confusion_matrix_path)
print("Saved history:", history_path)
print("Saved summary:", summary_path)
